# 🔴 Solution: QLoRA Linear Layer

Quantized base weight + low-rank adapter for efficient fine-tuning

In [ ]:
import torch
import torch.nn as nn


In [ ]:
# ✅ SOLUTION

class QLoRALinear(nn.Module):
    """
    QLoRA: 带 LoRA 适配器的量化线性层。

    核心思想：
    - 基础权重以 int8 格式存储（模拟 4-bit 量化），配合 fp32 缩放因子
    - LoRA 适配器保持全精度，负责任务特定的微调
    - 前向传播时反量化基础权重，加上 LoRA 增量

    参数:
        in_features: 输入维度
        out_features: 输出维度
        rank: LoRA 的低秩维度
        alpha: LoRA 缩放系数
    """
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha

        # ---- 量化权重 (冻结，不参与梯度计算) ----
        # int8 格式，范围 [-127, 127]
        # requires_grad=False: 量化权重不需要梯度
        self.quantized_weight = nn.Parameter(
            torch.zeros(out_features, in_features, dtype=torch.int8),
            requires_grad=False
        )

        # ---- 每行缩放因子 ----
        # scale[i] = 原始权重第 i 行的最大绝对值 / 127
        # 反量化: W_fp = quantized_weight.float() * scale
        self.scale = nn.Parameter(torch.ones(out_features, 1))

        # ---- LoRA 适配器 ----
        # LoRA: ΔW = A @ B, 其中 A: (in, rank), B: (rank, out)
        # A 初始化为小随机值, B 初始化为零 → 训练开始时 ΔW = 0
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

    def set_weight(self, W_fp32):
        """
        将浮点权重量化为 int8 存储。

        量化公式:
            scale = max(|W|, dim=1) / 127
            quantized = round(W / scale).clamp(-127, 127)
        """
        # 每行的最大绝对值, shape: (out, 1)
        # clamp(min=1e-8) 防止除以 0
        scale = W_fp32.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8) / 127.0

        # 量化: 除以 scale → 四舍五入 → 截断到 [-127, 127]
        q = (W_fp32 / scale).round().clamp(-127, 127).to(torch.int8)

        self.quantized_weight.data.copy_(q)
        self.scale.data.copy_(scale)

    def forward(self, x):
        """
        前向传播: 反量化基础输出 + LoRA 增量

        x: (B, in_features) → (B, out_features)
        """
        # ---- Step 1: 反量化 ----
        # int8 → float → 乘以 scale 恢复原始量级
        W_fp = self.quantized_weight.float() * self.scale  # (out, in)

        # ---- Step 2: 基础输出 ----
        # x @ W^T: (B, in) @ (in, out) = (B, out)
        base = x @ W_fp.T

        # ---- Step 3: LoRA 增量 ----
        # Δoutput = x @ A @ B * (alpha / rank)
        # alpha/rank 控制 LoRA 更新的幅度
        # rank 越大，alpha/rank 越小，单个参数的影响越小
        delta = x @ self.lora_A @ self.lora_B * (self.alpha / self.rank)

        # ---- Step 4: 合并 ----
        return base + delta

In [ ]:
torch.manual_seed(0)
in_f, out_f, rank = 64, 32, 4
layer = QLoRALinear(in_f, out_f, rank, alpha=2.0)

# Set weight from a full-precision reference
W_ref = torch.randn(out_f, in_f)
layer.set_weight(W_ref)

x = torch.randn(8, in_f)
y_qlora = layer(x)
y_ref = x @ W_ref.T  # full-precision baseline (no LoRA delta)

print("Output shape:", y_qlora.shape)          # (8, 32)
print("Max abs error vs fp32:", (y_qlora - y_ref).abs().max().item())

In [ ]:
from torch_judge import check
check("qlora")

## 📝 核心思路总结

1. **QLoRA = 量化 + LoRA**：基础权重用 int8 存储（模拟 4-bit），LoRA 适配器保持 fp32，兼顾内存效率和微调质量。
2. **量化公式**：`scale = max(|W|)/127`，`q = round(W/scale).clamp(-127,127)`，每行独立缩放，保留相对精度。
3. **LoRA 的零初始化**：B 初始化为零，训练开始时 ΔW=0，模型行为与纯量化模型一致，渐进学习任务特定信息。
4. **alpha/rank 缩放**：`alpha/rank` 控制 LoRA 更新的幅度，rank 越大单个参数影响越小，防止低秩更新过大扰动基础模型。